# Harvard's Endowment

## HBS Case
### *The Harvard Management Company and Inflation-Indexed Bonds*

$$
\newcommand{\mutarg}{\tilde{\boldsymbol{\mu}}^{\text{port}}}
\newcommand{\mux}{\tilde{\mu}}
\newcommand{\wEW}{\boldsymbol{\text{w}}^{\text{EW}}}
\newcommand{\wREG}{\boldsymbol{\text{w}}^{\text{REG}}}
\newcommand{\wRP}{\boldsymbol{\text{w}}^{\text{RP}}}
\newcommand{\wtan}{\boldsymbol{\text{w}}^{\text{tan}}}
$$

# 1. READING: HMC's Approach

### 1. 
There are thousands of individual risky assets in which HMC can invest.  Explain why MV optimization across 1,000 securities is infeasible.

### 2.
Rather than optimize across all securities directly, HMC runs a two-stage optimization.
1. They build asset class portfolios with each one optimized over the securities of the specific asset class.  
2. HMC combines the asset-class portfolios into one total optimized portfolio.

In order for the two-stage optimization to be a good approximation of the full MV-optimization on all assets, what must be true of the partition of securities into asset classes?

### 3.
Should TIPS form a new asset class or be grouped into one of the other 11 classes?

### 4. 
Why does HMC focus on real returns when analyzing its portfolio allocation? Is this just a matter of scaling, or does using real returns versus nominal returns potentially change the MV solution?

### 5.
The case discusses the fact that Harvard places bounds on the portfolio allocation rather than implementing whatever numbers come out of the MV optimization problem.

How might we adjust the stated optimization problem in the lecture notes to reflect the extra constraints Harvard is using in their bounded solutions given in Exhibits 5 and 6?

### 6. 
Exhibits 5 shows zero allocation to domestic equities and domestic bonds across the entire computed range of targeted returns, (5.75% to 7.25%). Conceptually, why is the constraint binding in all these cases? What would the unconstrained portfolio want to do with those allocations and why?

### 7.
Exhibit 6 changes the constraints, (tightening them in most cases.) How much deterioration do we see in the mean-variance tradeoff that Harvard achieved?

## Notes/ Answers (Not Exaustive)

1. MV Optimization across 1000 assets would be infeasiable mainly because of estimation. This would requrie 1000 expected returns and variances as well a large magnitude of covariances. With limited historical data the sample covariance matrix can not be inverted and that makes the estimates even more noisy. Since baskets of securities can be highly correlated the covariance matrix is nearly signular. Here the optimizer would amplify errors into extreme long/ short positions (without the bounds that Harvard sets). 

2. The securities must have similar return profiles and their returns (not risk necessarly) must be highly correlated. Further between each basket the correlation must be low and partition the assets well. The two stage approach does ignore security-level covariances across classes, so it works only if those covariances are small. If I can speculate on the use of the "Absolute Return" class for hedge funds is that these do not fit cleanly into any basket and these stratigies should have a cross-class covariance that is zero. 

3. It likely should be a new asset class as it measures CPI and consumer spending. While an argument could be made to share this section with domestic bonds (correlation of 0.5) the rest are 0.3 or low. TIPS also have much lower volatility (3% vs. 7%) and the opposite response to unexpected inflation, which hurts nominal bonds but is passed through to TIPS. The MV results confirm they're distinct in Exhibit 5 where we get 43 to 70% in TIPS and none in domestic bonds. 

4. We need real returns as these are funding a university where inflation will be a real cost.

5. We will use max and min constraints in our otpimization. We can gradually change these until we do not see any extreme allocations

6. The optimizer would like to hold less than 0% of these but is constrained as we can not short. MV weights depend on an asset's return relative to its covariance with the rest of the portfolio, not its standalone Sharpe ratio. Domestic bonds are dominated by TIPS, which offer similar return at a lower volitility with a 0.5 correlation so shorting bonds hedges TIPS' rate risk and funds a larger TIPS position. 

7. The deterioration is modest. At each target return, Exhibit 6's volatility is about 0.55–0.75 points higher than Exhibit 5's and Sharpe falls from .38 to .35. They only give up about 0.2 points of real return per year. 

***

# 2 Mean-Variance Optimization

### Data
You will need the file in the github repo, `data/multi_asset_etf_data.xlsx`.
- The time-series data gives monthly returns for the 11 asset classes and a short-term Treasury-bill fund return, (`SHV`.)
- The case does not give time-series data, so this data has been compiled outside of the case, and it intends to represent the main asset classes under consideration via various ETFs. For details on the specific securities/indexes, check the “Info” tab of the data.

### Excess Returns
- We consider `SHV` as the risk-free asset.
- We are going to analyze the problem in terms of **excess** returns, where `SHV` has been subtracted from the other columns.
- The risk-free rate changes over time, but the assumption is that investors know it’s value one-period ahead of time. Thus, at any given point in time, it is a risk-free rate for the next period. (This is often discussed as the "bank account" or "money market account" in other settings.)

### Adjustment
For ease of analysis, drop `QAI` from the dataset. Analyze the remaining 10 assets.

### Not Considered
- These are nominal returns-they are not adjusted for inflation, and in our calculations we are not making any adjustment for inflation.
- The exhibit data that comes via Harvard with the case is unnecessary for our analysis.

### Format
In the questions below, **annualize the statistics** you report.
- Annualize the mean of monthly returns with a scaling of 12.
- Annualize the volatility of monthly returns with a scaling of $\sqrt{12}$
- Note that we are not scaling the raw timeseries data, just the statistics computed from it (mean, vol, Sharpe).

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

#Prices Sheet of the data
Prices = pd.read_excel('FINM Portfolio/multi_asset_etf_data.xlsx', sheet_name = 'prices', index_col = 0, parse_dates=True)

#Total Returns 
Total_Rets = pd.read_excel('FINM Portfolio/multi_asset_etf_data.xlsx', sheet_name = 'total returns', index_col = 0, parse_dates=True)

#Excess Returns
Excess_Rets = pd.read_excel('FINM Portfolio/multi_asset_etf_data.xlsx', sheet_name = 'excess returns', index_col = 0, parse_dates=True)

Matplotlib is building the font cache; this may take a moment.


### 1. Summary Statistics
* Calculate and display the mean and volatility of each asset’s excess return. (Recall we use volatility to refer to standard deviation.)
* Which assets have the best and worst Sharpe ratios? 

Recall that the Sharpe Ratio is simply the ratio of the mean-to-volatility of excess returns:

$$\text{sharpe ratio of investment }i = \frac{\mux_i}{\sigma_i}$$

Be sure to annualize all three statss (mean, vol, Sharpe).



In [6]:
#QAI already dropped for us

FREQ = 12 # Monthly Data

stats = pd.DataFrame({'Mean' : Excess_Rets.mean() * FREQ,
                      'Vol' : Excess_Rets.std() * np.sqrt(FREQ),})

stats['Sharpe'] = stats['Mean'] / stats['Vol']

stats.sort_values('Sharpe', ascending = False)

,Mean,Vol,Sharpe
SPY,0.126955,0.141398,0.897856
HYG,0.034500,0.073326,0.470501
EFA,0.063754,0.148590,0.429057
IYR,0.070095,0.165531,0.423456
QAI,0.018659,0.049876,0.374120
PSP,0.076428,0.211323,0.361666
TIP,0.012776,0.049748,0.256817
EEM,0.044426,0.179085,0.248074
IEF,0.008596,0.062120,0.138377
DBC,0.013000,0.172276,0.075461


By absolute value we see that SPY has the best sharpe ratio delivering the most excess returns per unit of volitility. BWS has the lowest and is negitive average excess return which means that over the sample it underperformed the risk-free rate while still carrying meaningful volatility. 

***

# 3. Allocations

* Continue with the same data file as the previous section.

* Suppose the investor has a targeted mean excess return (per month) of $\mutarg$ = 0.01.

Build the following portfolios:







#### Equally-weighted (EW)
Rescale the entire weighting vector to have target mean $\mutarg$. Thus, the $i$ element of the weight vector is,

$$\wEW_i = \frac{1}{n}$$

#### “Risk-parity” (RP)
Risk-parity is a term used in a variety of ways, but here we have in mind setting the weight of the portfolio to be proportional to the inverse of its full-sample variance estimate. Thus, the $i$ element of the weight vector is,

$$\boldsymbol{\text{w}}^{\text{RP}}_i = \frac{1}{\sigma_i^2}$$


#### Mean-Variance (MV)
As described in `Section 2`.

### Comparing

In order to compare all these allocation methods, rescale each weight vector, such that it has targeted mean return of $\mutarg$.

* Calculate the performance of each of these portfolios over the sample.
* Report their mean, volatility, and Sharpe ratio. 
* How does performance compare across allocation methods?

In [ ]:
Rets = Excess_Rets.drop(columns = 'QAI')
TARGET = 0.01               # targeted mean excess return per month
mu = Rets.mean()            # monthly mean excess returns
Sigma = Rets.cov()          # monthly covariance matrix
n = len(mu)


# Raw (unscaled) weights for each method
w_raw = pd.DataFrame({
    'EW' : np.ones(n) / n,                          # equal weight: 1/n
    'RP' : 1 / Rets.var(),                          # risk parity: 1/variance
    'MV' : np.linalg.inv(Sigma)@mu,              # tangency: Sigma^-1 * mu
}, index = mu.index)

# Rescale each column so the portfolio mean (w'mu) equals the target
w = w_raw * (TARGET / (mu @ w_raw))

Port_Rets = Rets @ w

perf = pd.DataFrame({'Mean' : Port_Rets.mean() * FREQ, 'Vol'  : Port_Rets.std() * np.sqrt(FREQ)})
perf['Sharpe'] = perf['Mean'] / perf['Vol']
display((w * 100).round(1))    # weights in %
display(perf)

,EW,RP,MV
BWX,27.7,67.3,-87.6
DBC,27.7,15.2,2.7
EEM,27.7,14.1,-0.1
EFA,27.7,20.4,13.7
HYG,27.7,84.0,15.1
IEF,27.7,117.0,97.9
IYR,27.7,16.5,-16.6
PSP,27.7,10.1,-30.5
SPY,27.7,22.6,93.8
TIP,27.7,182.4,-14.2


,Mean,Vol,Sharpe
EW,0.12,0.275296,0.435894
RP,0.12,0.326142,0.367938
MV,0.12,0.081314,1.475763


In [12]:
print(w_raw)

      EW           RP         MV
BWX  0.1  1789.402347 -15.890598
DBC  0.1   404.323690   0.489016
EEM  0.1   374.164623  -0.013148
EFA  0.1   543.499678   2.489898
HYG  0.1  2231.829387   2.745142
IEF  0.1  3109.670560  17.764699
IYR  0.1   437.949880  -3.012893
PSP  0.1   268.712361  -5.530725
SPY  0.1   600.199058  17.015115
TIP  0.1  4848.721511  -2.586143


***

# 4. EXTRA: Out-of-Sample Performance

### 1. One-step Out-of-Sample (OOS) Performance
Let’s divide the sample to both compute a portfolio and then check its performance out of sample.
* Using only data through the end of `2023`, compute the weights built in Section 3.
* Rescale the weights, (using just the in-sample data,) to set each allocation to have the same mean return of $\mutarg$.
* Using those weights, calculate the portfolio’s Sharpe ratio within that sample.
* Again using those weights, (derived using data through `2023`,) calculate the portfolio’s OOS Sharpe ratio, which is based only on performance in `2024-2025`.

### 2. Rolling OOS Performance

Iterate the Out-of-Sample performance every year, not just the final year. Namely,
* Start at the end of `2015`, and calculate the weights through that time. Rescale them using the mean returns through that time.
* Apply the weights to the returns in the upcoming year, (`2016`.)
* Step forward a year in time, and recompute.
* Continue until again calculating the weights through `2023` and applying them to the returns in `2024-2025`.

Report the mean, volatility, and Sharpe from this dynamic approach for the following portfolios:
* mean-variance (tangency)
* equally-weighted
* risk-parity
* regularized

***

# 5. EXTRA: Without a Riskless Asset

Re-do Section 2 above, but in the model without a risk-free rate.

That is, build the MV allocation using the two-part formula in the `Mean-Variance` section of the notes.
* This essentially substitutes the risk-free rate with the minimum-variance portfolio.
* Now, the allocation depends nonlinearly on the target mean return, $\mutarg$. (With a risk-free rate, we simply scale the weights up and down to achieve the mean return.)

You will find that, conceptually, the answers are very similar.

***

# 6. EXTRA: Bayesian Allocation

Add the following allocation among the choices in `Section 3`...

#### Regularized (REG)
Much like the Mean-Variance portfolio, set the weights proportional to 

$$\wREG \sim \widehat{\Sigma}^{-1}\mux$$

but this time, use a regularized covariance matrix,

$$\widehat{\Sigma} = \frac{\Sigma + \Sigma_D}{2}$$

where $\Sigma_D$ denotes a *diagonal* matrix of the security variances, with zeros in the off-diagonals.

Thus, $\widehat{\Sigma}$ is obtained from the usual covariance matrix, $\Sigma$, but shrinking all the covariances to half their estimated values.

# 7. EXTRA: Inefficient Tangency

Return to analyzing the excess returns, as in `Section 3`. Include the hedge-fund ETF, QAI.

You might find that the tangency portfolio has a negative mean return. 
- It is on the inefficient portion of the MV frontier.

Calculate the optimal allocation by shorting the tangency portfolio. (That is, multiply the tangency weights by negative one to determine the allocation weights.)

Re-do the analysis of Section 3. Does the addition of `QAI` change much.